# Redes Neuronales con Pytorch

## Introduction

En este ejercicio se implementa una red neuronal para reconocimiento de digitos utilizando pytorch.

Antes de empezar la ejecución de las partes de codigo correspondienters a los ejercicios, se requiere importar todas las librerias necesarias.

In [1]:
# utilizado para la manipulación de directorios y rutas
import os
# Cálculo científico y vectorial para python
import numpy as np
# Libreria para graficos
from matplotlib import pyplot as plt


import torch
import torchvision # torch package for vision related things
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
from torch import nn  # All neural network modules
from torch.utils.data import DataLoader  # Gives easier dataset managment by creating mini batches etc.
from tqdm import tqdm  # For nice progress bar!

# le dice a matplotlib que incruste gráficos en el cuaderno
%matplotlib inline

In [16]:
# Here we create our simple neural network. For more details here we are subclassing and
# inheriting from nn.Module, this is the most general way to create your networks and
# allows for more flexibility. I encourage you to also check out nn.Sequential which
# would be easier to use in this scenario but I wanted to show you something that
# "always" works.
class NNEMNIST(nn.Module):
    def __init__(self, input_size, num_classes):
        super(NNEMNIST, self).__init__()
        # Our first linear layer take input_size, in this case 784 nodes to 50
        # and our second linear layer takes 50 to the num_classes we have, in
        # this case 10.
        self.fc1 = nn.Linear(input_size, 300)
        self.fc2 = nn.Linear(300, 100)
        self.fc3 = nn.Linear(100, num_classes)

        # self.fc1 = nn.Linear(input_size, 47)

    def forward(self, x):
        """
        x here is the mnist images and we run it through fc1, fc2 that we created above.
        we also add a ReLU activation function in between and for that (since it has no parameters)
        I recommend using nn.functional (F)
        """
        x = self.fc1(x)
        x = torch.relu(x)
        # x = F.sigmoid(self.fc1(x))
        x = self.fc2(x)
        x = torch.sigmoid(x)
        x = self.fc3(x)
        return x

In [22]:
# Set device cuda for GPU if it's available otherwise run on the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Hyperparameters of our neural network which depends on the dataset, and
# also just experimenting to see what works well (learning rate for example).
input_size = 784
num_classes = 47
learning_rate = 0.001
batch_size = 50000
num_epochs = 5

cuda


In [23]:
# Load Training and Test data
train_dataset = datasets.EMNIST(root="dataset/", split="bymerge", train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.EMNIST(root="dataset/", split="bymerge", train=False, transform=transforms.ToTensor(), download=True)
print(len(train_dataset))
print(len(test_dataset))

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

697932
116323


In [12]:
for c in train_dataset:
  print(c)

Se han truncado las últimas 5000 líneas del flujo de salida.
         [0.0000, 0.0000, 0.0000, 0.0039, 0.3569, 0.8588, 0.9647, 0.6431,
          0.0863, 0.0039, 0.0078, 0.3216, 0.9882, 0.8510, 0.0157, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1804, 0.8196,
          0.9333, 0.3686, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.1333, 0.9529, 0.9804, 0.3725, 0.0353,
          0.0000, 0.0000, 0.0000, 0.1451, 0.9804, 0.8510, 0.0157, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0157, 0.4549,
          0.9804, 0.5490, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0118, 0.3294, 0.9804, 0.7412, 0.0784, 0.0000,
          0.0000, 0.0000, 0.0000, 0.1255, 0.9608, 0.8510, 0.0157, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2000,
          0.9804, 0.8000, 0.0157, 0.0000],
         [0.0000, 0.0000, 0.1333, 0.8000, 0.9137, 0.3216, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0196, 0.8510, 0.9137, 0.0824, 0.00

KeyboardInterrupt: 

In [24]:
# Initialize network
model = NNEMNIST(input_size=input_size, num_classes=num_classes).to(device)

In [26]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [27]:
# Train Network
for epoch in range(num_epochs):
    train_loss, train_acc = [], []

    # bar = tqdm(dataloader['train'])
    bar = tqdm(train_loader)
    for batch_idx, (data, targets) in enumerate(bar):
    # for batch_idx, (data, targets) in enumerate(tqdm(train_loader)):
    # for batch_idx, (data, targets) in enumerate(train_loader):

        # data, targets = batch.data, batch.targets
        # data, targets = data.to(device), targets.to(device)

        # Get data to cuda if possible
        data = data.to(device=device)
        targets = targets.to(device=device)

        # print(data.shape)
        # Get to correct shape
        data = data.reshape(data.shape[0], -1)
        # print(data.shape)
        # print("-"*30)
        # forward
        scores = model(data)
        print(scores.shape)
        print(targets.shape)
        loss = criterion(scores, targets)

        # backward
        optimizer.zero_grad()
        loss.backward()

        # gradient descent or adam step
        optimizer.step()
        train_loss.append(loss.item())
        acc = (targets == torch.argmax(scores, axis=1)).sum().item() / len(targets)
        train_acc.append(acc)
        bar.set_description(f"loss {np.mean(train_loss):.5f} acc {np.mean(train_acc):.5f}")


loss 3.94344 acc 0.01992:   7%|▋         | 1/14 [00:04<00:57,  4.41s/it]

torch.Size([50000, 47])
torch.Size([50000])


loss 3.91454 acc 0.02051:  14%|█▍        | 2/14 [00:08<00:53,  4.49s/it]

torch.Size([50000, 47])
torch.Size([50000])


loss 3.88796 acc 0.03358:  21%|██▏       | 3/14 [00:13<00:51,  4.65s/it]

...

torch.Size([50000, 47])
torch.Size([50000])


loss 2.21704 acc 0.50039:  93%|█████████▎| 13/14 [01:00<00:04,  4.50s/it]

torch.Size([50000, 47])
torch.Size([50000])


loss 2.20791 acc 0.50222: 100%|██████████| 14/14 [01:05<00:00,  4.64s/it]

torch.Size([47932, 47])
torch.Size([47932])


In [ ]:
# Check accuracy on training & test to see how good our model
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()

    predicciones = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)
            x = x.reshape(x.shape[0], -1)

            scores = model(x)
            _, predictions = scores.max(1)
            predicciones.append(predictions)

            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)

    model.train()
    return num_correct/num_samples, predicciones

p_train, pred_train  = check_accuracy(train_loader, model)
p_test, pred_test  = check_accuracy(test_loader, model)

print(f"Accuracy on training set: {p_train*100:.2f}")
print(f"Accuracy on test set: {p_test*100:.2f}")

Accuracy on training set: 52.78
Accuracy on test set: 52.64
